In [9]:
import pandas as pd
import os

In [10]:
os.makedirs("../data/transformed", exist_ok=True)

In [11]:
cleaned_path = "../data/cleaned/online_retail_cleaned.csv"
df = pd.read_csv(cleaned_path)

In [12]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Add derived date/time columns
df['year'] = df['InvoiceDate'].dt.year
df['month'] = df['InvoiceDate'].dt.month
df['day'] = df['InvoiceDate'].dt.day
df['weekday'] = df['InvoiceDate'].dt.day_name()
df['InvoiceYear'] = df['InvoiceDate'].dt.year
df['InvoiceMonth'] = df['InvoiceDate'].dt.month
df['InvoiceDay'] = df['InvoiceDate'].dt.day
df['Weekday'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour

# Compute Revenue and Weekly Moving Average
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['WeeklyAvg'] = df.groupby('CustomerID')['Revenue'].transform(lambda x: x.rolling(7, min_periods=1).mean())


In [ ]:
# Create Date Dimension Table
date_dim = df[['InvoiceDate', 'year', 'month', 'day', 'weekday']].drop_duplicates().copy()
date_dim['is_weekend'] = date_dim['weekday'].isin(['Saturday', 'Sunday'])
date_dim.rename(columns={'InvoiceDate': 'date_id'}, inplace=True)
date_dim['date_id'] = pd.to_datetime(date_dim['date_id']).dt.date  # <-- IMPORTANT

# Create Product Dimension Table
product_dim = df[['StockCode', 'Description', 'UnitPrice']].drop_duplicates().copy()
product_dim.rename(columns={
    'StockCode': 'product_id',
    'Description': 'product_name',
    'UnitPrice': 'unit_price'
}, inplace=True)

# Create Sales Fact Table
sales_fact = df[['InvoiceNo', 'StockCode', 'InvoiceDate', 'Quantity', 'Revenue']].copy()
sales_fact.rename(columns={
    'InvoiceNo': 'invoice_no',
    'StockCode': 'product_id',
    'InvoiceDate': 'date_id'
}, inplace=True)
sales_fact['invoice_no'] = sales_fact['invoice_no'].astype(str) 
sales_fact['date_id'] = pd.to_datetime(sales_fact['date_id']).dt.date

# Save to CSV
sales_fact.to_csv("../data/transformed/sales_fact.csv", index=False)
product_dim.to_csv("../data/transformed/product_dim.csv", index=False)
date_dim.to_csv("../data/transformed/date_dim.csv", index=False)

print("✅ Transformed CSVs saved in data/transformed/")


✅ Transformed CSVs saved in data/transformed/
